In [3]:
!nvidia-smi

Mon May 11 13:32:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# import dependencies
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import html
import time

In [5]:

# function to convert the JavaScript object into an OpenCV image
def js_to_image(js_reply):
  """
  Params:
          js_reply: JavaScript object containing image from webcam
  Returns:
          img: OpenCV BGR image
  """
  # decode base64 image
  image_bytes = b64decode(js_reply.split(',')[1])
  # convert bytes to numpy array
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  # decode numpy array into OpenCV BGR image
  img = cv2.imdecode(jpg_as_np, flags=1)

  return img

In [6]:
# function to convert OpenCV Rectangle bounding box image into base64 byte string to be overlayed on video stream
def bbox_to_bytes(bbox_array):
  """
  Params:
          bbox_array: Numpy array (pixels) containing rectangle to overlay on video stream.
  Returns:
        bytes: Base64 image byte string
  """
  # convert array into PIL image
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  iobuf = io.BytesIO()
  # format bbox into png for return
  bbox_PIL.save(iobuf, format='png')
  # format return string
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))

  return bbox_bytes

In [7]:
# JavaScript to properly create our live video stream using our webcam as input
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "Status:";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML =
          '' +
          'When finished, click here or on the video to stop this demo';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; //video.videoWidth;
      captureCanvas.height = 480; //video.videoHeight;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }

      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }

      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

In [8]:
!git clone https://github.com/ultralytics/yolov5

Cloning into 'yolov5'...
remote: Enumerating objects: 17968, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 17968 (delta 77), reused 33 (delta 31), pack-reused 17863 (from 3)
Receiving objects: 100% (17968/17968), 17.11 MiB | 20.00 MiB/s, done.
Resolving deltas: 100% (12221/12221), done.


In [9]:
import os
import sys

In [10]:
!ls yolov5

benchmarks.py	 data	     LICENSE	     README.zh-CN.md   train.py
CITATION.cff	 detect.py   models	     requirements.txt  tutorial.ipynb
classify	 export.py   pyproject.toml  segment	       utils
CONTRIBUTING.md  hubconf.py  README.md	     tests	       val.py


In [11]:
!cd yolov5

In [12]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.6 MB/s eta 0:00:00


In [13]:
import torch
model = torch.hub.load('/content/yolov5', 'custom', path='/content/yolov5s.pt', source='local')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['urllib3>=2.6.0 ; python_version > "3.8"'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ requirements: ❌ Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6

YOLOv5 🚀 v7.0-484-g70b964b6 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

100%|██████████| 14.1M/14.1M [00:00<00:00, 112MB/s] 

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [14]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import cv2
from google.colab.patches import cv2_imshow


In [15]:
video_stream()
label_html = 'Capturing...'
bbox = ''
count = 0
while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break
    img = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480,640,4], dtype=np.uint8)
    detections = model(img[..., ::-1])
    results = detections.pandas().xyxy[0].to_dict(orient="records")
    for result in results:
      con = result['confidence']
      cs  =str( result['name'])
      x1  = int(result['xmin'])
      y1  = int(result['ymin'])
      x2  = int(result['xmax'])
      y2  = int(result['ymax'])

      cv2.rectangle(bbox_array,(x1, y1), (x2, y2),(255,0,0),2)
      cv2.putText(bbox_array, "{} [{:.2f}]".format(cs, float(con)),
                        (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (255,0,0),2)

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes

<IPython.core.display.Javascript object>

In [25]:
# Inisialisasi variabel pendukung
w = 640 # Asumsi lebar frame
video_stream()
label_html = 'Capturing... (Tekan S di Keyboard Browser)'
bbox = ''

# Tambahkan JavaScript untuk menangkap tombol 'S' di browser
display(Javascript('''
  var s_pressed = false;
  document.addEventListener('keydown', (e) => {
    if (e.key === 's' || e.key === 'S') {
      s_pressed = true;
    }
  });

  // Fungsi untuk mengecek status tombol ke Python
  window.checkKey = function() {
    var state = s_pressed;
    s_pressed = false; // Reset setelah dibaca
    return state;
  };
'''))
count = 0


while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break
     # Cek apakah tombol 'S' ditekan via JS
    save_now = eval_js('window.checkKey()')
    img = js_to_image(js_reply["img"])
    image = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480,640,4], dtype=np.uint8)
    detections = model(img[..., ::-1])
    results = detections.pandas().xyxy[0].to_dict(orient="records")

    for result in results:
      con = result['confidence']
      cs  =str( result['name'])
      x1  = int(result['xmin'])
      y1  = int(result['ymin'])
      x2  = int(result['xmax'])
      y2  = int(result['ymax'])
      center = int((x1 + x2)/2), int((y1 + y2)/2)
      ## Menghitung Jarak ##
      eyeradius = w / 20
      yeye = y1 + y2/3
      reye = x1 + (x2/2) - (x2/5)
      leye = x1 + (x2/2) + (x2/5)
      space = leye - reye
      f = 690
      r = 10
      distance = f * r / space
      distance_in_cm = int(distance)
      if distance_in_cm < 25:
            status = "Sangat Dekat"
            warna = (0, 0, 255) # Merah
      elif 25 <= distance_in_cm <= 50:
            status = "Dekat"
            warna = (0, 165, 255) # Orange
      elif 51 <= distance_in_cm <= 100:
            status = "Sedang"
            warna = (0, 255, 255) # Kuning
      else:
            status = "Jauh"
            warna = (0, 255, 0)
      cv2.putText(bbox_array, str(distance_in_cm) +"cm", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255),2)
      #cv2.rectangle(bbox_array,(x1, y1), (x2, y2),(255,0,0),2)
      cv2.rectangle(bbox_array, (x1, y1), (x2, y2), warna, 2)

      if save_now:
        img_name = 'gbr.png'
        cv2.imwrite(img_name, image)
        img = cv2.imread(img_name)
        detections = model(img)
        results = detections.pandas().xyxy[0].to_dict(orient="records")
        sangat_dekat_count = 0  # < 25 cm
        dekat_count = 0         # 25 - 50 cm
        sedang_count = 0        # 51 - 100 cm
        jauh_count = 0          # > 100 cm
        for result in results:
          con = result['confidence']
          cs  =str( result['name'])
          x1  = int(result['xmin'])
          y1  = int(result['ymin'])
          x2  = int(result['xmax'])
          y2  = int(result['ymax'])
          eyeradius = w / 20
          yeye = y1 + y2/3
          reye = x1 + (x2/2) - (x2/5)
          leye = x1 + (x2/2) + (x2/5)
          space = leye - reye
          f = 690
          r = 10
          distance = f * r / space
          distance_in_cm = int(distance)
          cv2.putText(img, str(distance_in_cm) +"cm", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255),1)
          # 2. Logika Tambah Nilai jika jarak < 20
          if distance_in_cm < 25:
                        sangat_dekat_count += 1
                        status = "Sangat Dekat"
                        warna = (0, 0, 255) # Merah
          elif 25 <= distance_in_cm <= 50:
                        dekat_count += 1
                        status = "Dekat"
                        warna = (0, 165, 255) # Orange
          elif 51 <= distance_in_cm <= 100:
                        sedang_count += 1
                        status = "Sedang"
                        warna = (0, 255, 255) # Kuning
          else:
                        jauh_count += 1
                        status = "Jauh"
                        warna = (0, 255, 0) # Hijau

                    # Tampilkan Status di dekat objek
          cv2.putText(img, status, (x1, y2 + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, warna, 1)
          cv2.rectangle(img, (x1, y1), (x2, y2), warna, 2)

        bg_color = (0, 0, 0)
        cv2.putText(img, f"S. Dekat (<25cm): {sangat_dekat_count}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        cv2.putText(img, f"Dekat (25-50cm): {dekat_count}", (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 2)
        cv2.putText(img, f"Sedang (51-100cm): {sedang_count}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
        cv2.putText(img, f"Jauh (>100cm): {jauh_count}", (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        print(   sangat_dekat_count , # < 25 cm
        dekat_count ,         # 25 - 50 cm
        sedang_count ,        # 51 - 100 cm
        jauh_count   )


        print(f"Gambar disimpan: {img_name}")
        #img_counter += 1
        #cv2.imshow('YOLOv5 Image Detection', img)
        cv2.imwrite(img_name, img)

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

0 1 0 0
Gambar disimpan: gbr.png
0 1 0 0
Gambar disimpan: gbr.png


In [41]:
import pandas as pd
import time

# Inisialisasi variabel pendukung
w = 640
video_stream()
label_html = 'Capturing... (Tekan S untuk Simpan ke Excel)'
bbox = ''
history_log = [] # List untuk menampung semua hasil deteksi

# JavaScript untuk menangkap tombol 'S'
display(Javascript('''
  var s_pressed = false;
  document.addEventListener('keydown', (e) => {
    if (e.key === 's' || e.key === 'S') {
      s_pressed = true;
    }
  });
  window.checkKey = function() {
    var state = s_pressed;
    s_pressed = false;
    return state;
  };
'''))

while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break

    save_now = eval_js('window.checkKey()')
    img = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480,640,4], dtype=np.uint8)

    # Deteksi Real-time
    detections = model(img[..., ::-1])
    results = detections.pandas().xyxy[0].to_dict(orient="records")

    sangat_dekat_count = 0
    dekat_count = 0
    sedang_count = 0
    jauh_count = 0

    for result in results:
        x1, y1, x2, y2 = int(result['xmin']), int(result['ymin']), int(result['xmax']), int(result['ymax'])

        # Logika Jarak (Fokus pada perhitungan space)
        space = (x1 + (x2/2) + (x2/5)) - (x1 + (x2/2) - (x2/5))
        distance_in_cm = int(690 * 10 / space) if space != 0 else 0

        if distance_in_cm < 25:
            sangat_dekat_count += 1
            warna = (0, 0, 255)
        elif 25 <= distance_in_cm <= 50:
            dekat_count += 1
            warna = (0, 165, 255)
        elif 51 <= distance_in_cm <= 100:
            sedang_count += 1
            warna = (0, 255, 255)
        else:
            jauh_count += 1
            warna = (0, 255, 0)

        cv2.rectangle(bbox_array, (x1, y1), (x2, y2), warna, 2)
        cv2.putText(bbox_array, f"{distance_in_cm}cm", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, warna, 2)

    # LOGIKA SIMPAN KE EXCEL SAAT TOMBOL 'S' DITEKAN
    if save_now:
        # 1. Simpan data ke dalam dictionary
        data_baru = {
            "Waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
            "Sangat Dekat (<25cm)": sangat_dekat_count,
            "Dekat (25-50cm)": dekat_count,
            "Sedang (51-100cm)": sedang_count,
            "Jauh (>100cm)": jauh_count,
            "Total Objek": len(results)
        }

        # 2. Tambahkan ke history list
        history_log.append(data_baru)

        # 3. Konversi list ke DataFrame Pandas dan simpan ke Excel
        df = pd.DataFrame(history_log) #
        file_name = 'laporan_deteksi.xlsx'
        df.to_excel(file_name, index=False) #

        #print(f"Data ke-{len(history_log)} berhasil disimpan ke {file_name}")
        #print(df)
    cv2.putText(bbox_array, f"SD : {df["Sangat Dekat (<25cm)"].sum()}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    cv2.putText(bbox_array, f"D : {df["Dekat (25-50cm)"].sum()}", (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 2)
    cv2.putText(bbox_array, f"S : {df["Sedang (51-100cm)"].sum()}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    cv2.putText(bbox_array, f"J : {df["Jauh (>100cm)"].sum()}", (20, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(bbox_array, f"TOT : {df["Total Objek"].sum()}", (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
    bbox = bbox_to_bytes(bbox_array)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>